# Lab 0b — The climate of New York City

<a class="btn btn-primary btn-sm" href="https://dhruvbalwada.github.io/intro-climate-modeling-fall2026/labs/lab0c_nyc_climate.ipynb" download>&#8681; Download this notebook (.ipynb)</a>

*Then upload it to [leap.2i2c.cloud](https://leap.2i2c.cloud/) and open it there.*

*Week 1, in class · Introduction to Climate Modeling*

Daily temperature in Central Park, 1995–2014. We'll use it to answer, concretely: **what is climate?** and **what is a model?** — and to build your *first* climate model (it has two parameters).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

url = "https://dhruvbalwada.github.io/intro-climate-modeling-fall2026/labs/data/centralpark_obs_era5_1995-2014.csv"
df = pd.read_csv(url, index_col=0, parse_dates=True)
df.head()

`t_obs` = the station thermometer (NOAA). `t_era5` = we'll come back to that one.

## 1. Weather vs climate

Plot two years of data. Then plot the histogram of **all 20 years**.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
df["t_obs"]["2010":"2011"].plot(ax=ax[0], lw=0.8, color="k")
ax[0].set_ylabel("°C"); ax[0].set_title("two years of WEATHER")
ax[1].hist(df["t_obs"].dropna(), bins=50, color="k", alpha=0.7)
ax[1].set_xlabel("°C"); ax[1].set_title("twenty years of CLIMATE (the distribution)")
plt.tight_layout()

print(f"mean = {df['t_obs'].mean():.1f} °C   std = {df['t_obs'].std():.1f} °C")
print(f"record low = {df['t_obs'].min():.1f} °C   record high = {df['t_obs'].max():.1f} °C")

**Discuss:** The mean and the histogram *are* NYC's climate (over this chosen 20-year window). What can this distribution tell you that the mean alone can't? *(Hint: which tail do heat waves live in?)*

## 2. Your first climate model — fit it by hand

Most of the wiggle is the **seasonal cycle**. Let's model temperature as:

$$T(d) \,=\, T_0 \,+\, A\,\cos\!\big(2\pi\,(d - 201)/365\big)$$

where $d$ = day of year, and the peak is pinned to July 20 (day 201). **This model has exactly two parameters:**

- $T_0$ = ______ (what does it mean? what are its units?)
- $A$ = ______ (what does it mean? units?)

**Your job: pick values for `T0` and `A` by hand.** Run the cell, look at the plot and the error, adjust, repeat. Stop when you can't do better.

In [ ]:
# EDIT THESE TWO NUMBERS, run, look, repeat:
T0 = 10.0    # °C
A  = 5.0     # °C

doy = df.index.dayofyear
df["t_model"] = T0 + A * np.cos(2 * np.pi * (doy - 201) / 365)

fig, ax = plt.subplots(figsize=(10, 3.5))
df["t_obs"]["2010":"2011"].plot(ax=ax, lw=0.8, color="k", label="observations")
df["t_model"]["2010":"2011"].plot(ax=ax, lw=2.5, color="tab:red", label=f"your model: T0={T0}, A={A}")
ax.set_ylabel("°C"); ax.legend()

rmse = np.sqrt(((df["t_obs"] - df["t_model"])**2).mean())
ax.set_title(f"your model's error (RMSE): {rmse:.2f} °C")
plt.tight_layout()

**When you're happy:** shout out your $T_0$, $A$, and RMSE — we'll collect them on the board.

> Everyone fit *the same data* with *the same model* — and got **different numbers**. That spread is **parameter uncertainty**, and you just experienced where it comes from. (A computer can minimize the error exactly — we'll let it, in Lab 1 — but its answer is just one more point in this cloud, privileged only by its error being lowest.)

**Also discuss:** your two-parameter model explains most of the variance. What physical fact makes the seasonal cycle so predictable, when next Tuesday's weather isn't?

## 3. Two more "NYCs": a reanalysis and a climate model

The same location, from **ERA5** (a weather model continuously corrected by observations) and from a **CMIP6 climate model** (free-running — it invents its own weather).

In [ ]:
url2 = "https://dhruvbalwada.github.io/intro-climate-modeling-fall2026/labs/data/centralpark_cmip6_MPI-ESM1-2-HR_1995-2014.csv"
cm = pd.read_csv(url2, index_col=0, parse_dates=True)
cm.index = cm.index.normalize()
df["t_cmip6"] = cm["t_cmip6"].reindex(df.index)

fig, ax = plt.subplots(figsize=(11, 3.5))
for c, col, lab in [("t_obs", "k", "station"), ("t_era5", "tab:blue", "ERA5"), ("t_cmip6", "tab:orange", "CMIP6 model")]:
    df[c]["2010-06":"2010-12"].plot(ax=ax, lw=1.2, color=col, label=lab)
ax.set_ylabel("°C"); ax.legend(); ax.set_title("six months, three versions of NYC")
plt.tight_layout()

doyg = df.index.dayofyear
a = df.apply(lambda c: c - c.groupby(doyg).transform("mean"))  # each series minus its own seasonal cycle
print("day-to-day correlation with the station (seasonal cycles removed):")
print(f"  ERA5  : r = {a['t_obs'].corr(a['t_era5']):.2f}")
print(f"  CMIP6 : r = {a['t_obs'].corr(a['t_cmip6']):.2f}")
print(f"\n20-year means: station {df['t_obs'].mean():.1f} °C · ERA5 {df['t_era5'].mean():.1f} °C · CMIP6 {df['t_cmip6'].mean():.1f} °C")

**Discuss (the big ones):**

1. ERA5 tracks the station day-by-day; the CMIP6 model completely ignores it. Why is that *not* a failure of the CMIP6 model? What *should* it get right?
2. All three 20-year means differ (station warmest, CMIP6 coldest). Give two *different kinds* of reasons why. *(One is about what a grid cell is; one is about the model being imperfect.)*
3. Your 2-parameter model has a lower RMSE at this station than the CMIP6 model. Is yours the better model? For what, and for what not?